# Local Setup Check — Running and Training Language Models on Your Own Machine

**ISM 6564 — Text Analytics · Fall 2026 · from Week 5 on**

From Week 5 the course runs language models on your own machine, and in Week 6 it fine-tunes
them. This notebook does two things. It walks you through setting up a Windows PC with an
NVIDIA GPU or a Mac with Apple silicon for that work, linking the official instructions at each
step. Then it **tests your installation**: it checks your Python, PyTorch and GPU, gets a small
language model to write a reply, trains it for ten steps, and ends with a PASS / WARN / FAIL
summary.

> ### Read this first
>
> - **This notebook is primarily a test of your installation.** Its job is to tell you
>   whether your machine is ready and, if it is not, exactly which part is failing.
> - **I am not tech support for your computer.** I cannot debug individual laptops, drivers
>   or Python installs. Getting your own machine working is your responsibility. Work from
>   the steps below, the official guides they link to and the exact error messages.
> - **Put in the time.** A clean install takes about an hour, most of it downloading. A
>   troubled one can take an evening. Start well before an assignment is due, not the night
>   before.
> - **You are never blocked.** Every course notebook also runs in Google Colab, which has a
>   free NVIDIA T4 GPU. If your install is not working when you need it, do the work in Colab
>   and keep fixing the install alongside.
> - **AI assistants are good at this.** Give yours the exact error message and the summary
>   block printed at the end of this notebook. Treat its answer as a lead to check against the
>   official page, especially before running anything that uninstalls or deletes.

## How to use this notebook

1. Find your machine under **Which path are you on?** and follow that path. If you built the
   course environment with `./setup.sh` in Week 1, you already have most of it; your path says
   what, if anything, to add.
2. Open this notebook in VS Code and select your course environment as the kernel (top right).
3. **Run All.** The checks never change your system. They read settings, download one small
   model (SmolLM2-135M-Instruct, about 270 MB, into the Hugging Face cache) and run it. Once the
   model is downloaded, the checks take under a minute (9 seconds on the instructor's M5 Max).
4. Read the summary at the end. Fix every **FAIL** from the top down, because a later check
   often fails as a result of an earlier one. Read every **WARN**, since some are fine for your
   machine. After any install, **restart the kernel** and run the notebook again.

## Which path are you on?

| Your machine | What PyTorch can use | Your path |
| --- | --- | --- |
| Mac with Apple silicon (M1, M2, M3, M4 or M5) | the Mac's own GPU, through Apple's Metal (`mps`) | **A** |
| Windows PC with an NVIDIA GPU (GeForce GTX / RTX, or an NVIDIA RTX workstation card) | the NVIDIA GPU, through CUDA (`cuda`) | **B**, or **C** (optional: Linux inside Windows) |
| Linux PC with an NVIDIA GPU | the NVIDIA GPU (`cuda`) | **C**, from step 3 |
| Intel Mac · Windows PC without an NVIDIA GPU (Intel or AMD graphics, or an Arm / Snapdragon laptop) | the CPU only | **D** |
| Chromebook, tablet, or a computer you cannot install software on | nothing local | **Colab** |

**How to tell.**

- **On a Mac:** Apple menu → *About This Mac*. "Chip: Apple M…" means Apple silicon;
  "Processor: … Intel …" means an Intel Mac. The macOS version is on the same screen.
- **On Windows:** Task Manager → *Performance*. Each *GPU* entry names its card, and one whose
  name starts with "NVIDIA" puts you on path B.

## Three facts behind most failures

1. **The notebook's kernel must be the Python you installed into.** Everything goes into one
   environment: either the course's `.venv`, built by `./setup.sh`, or a conda environment
   named `ism6564`. That same environment must be the kernel selected in VS Code. Most "I
   installed it and it still says `No module named torch`" problems come down to two different
   Pythons. Check 1 below prints the path of the Python this notebook is running on.
2. **The PyTorch build must match your hardware.** PyTorch comes in several builds:
   - **On Windows**, plain `pip install torch`, and the course's `./setup.sh`, install the
     **CPU-only** build. The NVIDIA builds come from PyTorch's own package index, one for each
     CUDA version (`cu126`, `cu130`, …).
   - **On a Mac**, the standard build already includes the Mac GPU backend.
   - **On Linux**, the standard build is an NVIDIA build for CUDA 13.
3. **An NVIDIA GPU needs a recent driver, not the CUDA Toolkit.** PyTorch's NVIDIA builds carry
   their own CUDA libraries. What they need from you is an NVIDIA **driver** new enough for their
   CUDA version. You do not need the CUDA Toolkit, cuDNN or Visual Studio to run or train models
   in this course. The toolkit is only for compiling your own GPU code.

## Path A — Mac with Apple silicon

Apple lists the requirements for the current PyTorch on a Mac: a Mac with Apple silicon,
**macOS 14.0 (Sonoma) or later**, Python 3.10 or later and the Xcode command-line tools
([Apple: Accelerated PyTorch training on Mac](https://developer.apple.com/metal/pytorch/)).

1. **Update macOS** to 14 or later: System Settings → General → Software Update.
2. **Install the command-line tools.** In Terminal, run `xcode-select --install`. If it says
   they are already installed, move on.
3. **Use an Apple-silicon Python, never an Intel one.** An Intel build of Python still runs on
   an M-series Mac, translated by Rosetta, but it cannot see the GPU. This is the most common
   Mac failure. It usually comes from an old Anaconda or Miniconda install, or from an
   environment that Migration Assistant copied over from an older Intel Mac. Check in Terminal,
   inside your course environment:

   ```bash
   python -c "import platform; print(platform.machine())"
   ```

   It must print `arm64`. If it prints `x86_64`, remove that Python distribution. Then either
   install the **Apple Silicon** Miniconda, or use `./setup.sh`, which downloads an
   Apple-silicon Python for you.
4. **Build the environment** in one of two ways:
   - **uv, the course default.** In your `6564F26-DATA` folder, run `./setup.sh`. The PyTorch it
     installs already includes the Mac GPU backend, so there is nothing to add.
   - **Miniconda.** The course's [getting-started guide](https://github.com/prof-tcsmith/6564F26-DATA/blob/main/docs/getting-started-local.md) walks through it. In short:

     ```bash
     conda create -n ism6564 python=3.12 -y
     conda activate ism6564
     pip install torch torchvision torchaudio
     ```

     Then install the packages listed under **After installing**, below.
5. **Check it.** In Terminal (uv: `.venv/bin/python` instead of `python`):

   ```bash
   python -c "import torch; print(torch.__version__, torch.backends.mps.is_available())"
   ```

   The line must end in `True`.

**What is different about a Mac**

- **The GPU shares the Mac's memory.** There is no separate graphics memory: the model, macOS
  and every open app share the same RAM. A model at full precision needs 4 bytes per learned
  number, so Qwen2.5-1.5B needs about 6.2 GB. Training needs about four times as much (see
  **Training on your own machine**, below). On an 8 GB or 16 GB Mac, close other apps before a
  big run.
- **A few PyTorch operations have no Mac GPU version yet.** Setting
  `PYTORCH_ENABLE_MPS_FALLBACK=1` *before* `import torch` runs them on the CPU instead of
  stopping with an error
  ([PyTorch: MPS environment variables](https://docs.pytorch.org/docs/stable/mps_environment_variables.html)).
  This notebook and the assignments set it for you.
- **Stay at full precision.** Half precision (`float16` or `bfloat16`) halves the memory. But on
  the instructor's M5 Max it made writing at most about 20% faster, and it turned
  Qwen2.5-0.5B's replies into nonsense. The course notebooks load models at full precision
  (`float32`).
- **Beware the fix an "out of memory" error suggests.** The message recommends
  `PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0`. PyTorch's documentation warns that this setting
  "may cause system failure" (the same page as above), because it removes the limit on how
  much memory the GPU may take. Use a smaller model instead.
- **MLX is optional.** It is Apple's own machine-learning framework, built for Apple silicon.
  - Its `mlx-lm` package runs and fine-tunes Hugging Face language models from the terminal,
    with LoRA and QLoRA ([mlx-lm](https://github.com/ml-explore/mlx-lm);
    [its fine-tuning guide](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md)).
  - On an M5 with macOS 26.2 or later, MLX uses the GPU's new Neural Accelerators
    ([Apple Machine Learning Research](https://machinelearning.apple.com/research/exploring-llms-mlx-m5)).
  - The course notebooks use PyTorch, so you do not need MLX for the course. To try it
    (uv: `uv pip install --python .venv mlx-lm`, then run the second line with
    `.venv/bin/mlx_lm.generate`):

    ```bash
    pip install mlx-lm
    mlx_lm.generate --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --prompt "In one sentence, what does a GPU do?"
    ```

  - On the instructor's M5 Max, this 4-bit copy of Qwen2.5-0.5B wrote about 570 tokens per
    second, against about 73 for the full-precision model in PyTorch. That is not a fair race
    (4-bit numbers against 32-bit ones), but it shows what quantization and Apple's own
    software can do.

## Path B — Windows with an NVIDIA GPU

1. **Update the NVIDIA driver.** Download the newest driver for your card from
   [nvidia.com/drivers](https://www.nvidia.com/en-us/drivers/) (Game Ready or Studio, either
   works), install it and restart.
2. **Check it.** In PowerShell, run `nvidia-smi`. It prints a table with your GPU's name. The
   **CUDA Version** in the top-right corner is the newest CUDA your *driver* can run. It is not
   something you installed, and you do not need to install it.
3. **Pick the PyTorch build** that fits your GPU and driver. For PyTorch 2.14, the current
   version, there are Windows builds for CUDA 12.6 (`cu126`), 13.0 (`cu130`) and 13.2
   (`cu132`). The selector at
   [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/) always has the
   current list.

   | Your GPU | "CUDA Version" in `nvidia-smi` | Use |
   | --- | --- | --- |
   | GeForce RTX 50-series | 13.0 or higher (driver 580 or newer) | `cu130` |
   | GeForce RTX 20-, 30- or 40-series, GTX 16-series, NVIDIA RTX workstation cards | 12.6 or higher | `cu126` (or `cu130`, with a driver of 580 or newer) |
   | GeForce GTX 10-series, 900-series, Titan V | 12.6 or higher | `cu126` only |

   - **RTX 50-series cards** need a PyTorch built for CUDA 12.8 or later
     ([PyTorch 2.7 release](https://pytorch.org/blog/pytorch-2-7/)), so `cu126` will not run
     on them.
   - **CUDA 13 dropped support** for cards older than the Turing generation (GTX 16 and RTX 20).
     The [CUDA release notes](https://docs.nvidia.com/cuda/cuda-toolkit-release-notes/index.html)
     record this and give the minimum driver for each CUDA version.
   - **To look up your card's generation** (its "compute capability"), see
     [NVIDIA's list](https://developer.nvidia.com/cuda-gpus). Check 3 below prints it too.
4. **Install the NVIDIA build of PyTorch.** Use `cu126` in the lines below, or `cu130` if the
   table says so.
   - **uv (you ran `./setup.sh`).** `setup.sh` installed the CPU-only build, because that is
     what Windows gets by default. In Git Bash or PowerShell, in your `6564F26-DATA` folder, run:

     ```bash
     uv pip install --python .venv --reinstall-package torch torch --index-url https://download.pytorch.org/whl/cu126
     ```

     Keep running notebooks from VS Code with the `.venv` kernel, as before.
     **`./setup.sh`, `uv sync` and `uv run` put the CPU-only build back**, because they restore
     the course's locked version. After any of them, run this line again.
   - **Miniconda.** In the Anaconda Prompt, run `conda activate ism6564` first, then:

     ```bash
     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
     ```

     If the CPU build is already installed, run `pip uninstall -y torch torchvision torchaudio`
     first.
5. **Check it.** In the same terminal (uv: `.venv\Scripts\python` instead of `python`):

   ```bash
   python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
   ```

   The version should end in `+cu126` or `+cu130`, followed by `True`. A version with no such
   ending, or one ending in `+cpu`, is still the CPU-only build.

**What else can go wrong on Windows**

- **`import torch` fails** with `OSError: [WinError 126] … Error loading "…\torch\lib\c10.dll"`
  (or another `.dll`). Install the latest **Microsoft Visual C++ Redistributable (x64)**
  ([Microsoft](https://learn.microsoft.com/en-us/cpp/windows/latest-supported-vc-redist)), then
  restart.
- **`pip` fails with a path error deep inside `site-packages`.** Windows has a 260-character
  path limit. Either enable long paths
  ([Microsoft](https://learn.microsoft.com/en-us/windows/win32/fileio/maximum-file-path-limitation))
  or keep your course folder short, such as `C:\ism6564`.
- **The model does not fit on the GPU.** GPU memory is separate from the PC's RAM, and laptop
  GPUs often have only 4–8 GB. At full precision Qwen2.5-1.5B needs about 6.2 GB of it. If a
  model does not fit, use a smaller one or Colab.
- **You do not need the CUDA Toolkit, cuDNN or Visual Studio.** Install the toolkit only if you
  later compile GPU code yourself
  ([NVIDIA's Windows guide](https://docs.nvidia.com/cuda/cuda-installation-guide-microsoft-windows/index.html)).

## Path C — Windows with an NVIDIA GPU, through WSL2 (optional)

WSL2 runs a real Linux inside Windows and shares your NVIDIA GPU with it. Much
machine-learning software is written for Linux first, so this path avoids most Windows-only
problems. In return, you work in a Linux terminal. Take this path only if you are comfortable
there; Path B is enough for this course.

1. **Update the Windows driver only.** Update the Windows NVIDIA driver as in Path B, step 1.
   **Do not install an NVIDIA driver inside Linux**, because WSL uses the Windows one. See
   [NVIDIA: CUDA on WSL](https://docs.nvidia.com/cuda/wsl-user-guide/index.html) and
   [Microsoft: CUDA in WSL](https://learn.microsoft.com/en-us/windows/ai/directml/gpu-cuda-in-wsl).
2. **Install WSL.** You need Windows 11, or Windows 10 version 2004 or later. In PowerShell,
   run as administrator, run `wsl --install`, restart, and create your Linux user
   ([Microsoft: install WSL](https://learn.microsoft.com/en-us/windows/wsl/install)).
3. **Check the GPU.** In the Ubuntu terminal, `nvidia-smi` should list your card. On a Linux
   PC, install your distribution's NVIDIA driver first; `nvidia-smi` is the same check.
4. **Build the environment inside Linux.** Clone `6564F26-DATA` into your Linux home folder
   (`~/`), not under `/mnt/c/…`, which is much slower, and run `./setup.sh`. On Linux the
   standard PyTorch is already an NVIDIA build, for CUDA 13. It needs a driver of 580 or newer
   and a GTX 16-series, RTX 20-series or newer card. For an older card or driver, swap in the
   `cu126` build with the uv line from Path B, step 4.
5. **Connect VS Code.** Install the **WSL** extension in VS Code on Windows, then run `code .`
   in your Linux course folder; VS Code reopens connected to Linux. Install the Python and
   Jupyter extensions there too ([VS Code: WSL](https://code.visualstudio.com/docs/remote/wsl)).

## Path D — No GPU that PyTorch can use

**Windows or Linux without an NVIDIA GPU, including Arm (Snapdragon) laptops.** Use the CPU.
- On Windows, `./setup.sh` already installs the CPU build.
- With Miniconda, run
  `pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu`.
- PyTorch does not publish Windows-on-Arm builds on PyPI, the normal `pip` index. If the install
  fails on an Arm laptop, use Colab.

The course's small models run on a CPU, only more slowly: Assignment 5's filled-in notebook took
95 seconds on the instructor's Mac CPU and will take longer on a laptop. Use Colab for anything
heavy.

**Intel Macs.** PyTorch stopped publishing Intel-Mac builds after version 2.2.2, and current
Hugging Face `transformers` needs PyTorch 2.5 or newer. The course's language-model notebooks
therefore will not install on an Intel Mac: use Colab.

## Colab — always available

Open a notebook in [Google Colab](https://colab.research.google.com/), then choose *Runtime →
Change runtime type → T4 GPU*. PyTorch, `transformers` and the NVIDIA driver are already
installed. Downloaded models are lost when the session ends, and an idle session is
disconnected after a while, so save your work often. You can run this notebook in Colab too, to
see what a working setup looks like.

## After installing: the other packages, and the notebook kernel

**uv (`./setup.sh`)** installed every package the course uses. There is nothing to add.

**Miniconda.** With `ism6564` active, install the rest, then register the environment as a
notebook kernel:

```bash
pip install transformers accelerate datasets peft sentence-transformers huggingface-hub scikit-learn pandas matplotlib jupyter ipykernel ipywidgets
python -m ipykernel install --user --name ism6564 --display-name "Python (ism6564)"
```

**VS Code.**
1. Install the **Python** and **Jupyter** extensions, both by Microsoft.
2. Open this notebook and click **Select Kernel** (top right).
3. Choose your environment: the `.venv` inside `6564F26-DATA` (uv) or **Python (ism6564)**
   (Miniconda). See
   [VS Code: Jupyter notebooks](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
   and [VS Code: Python environments](https://code.visualstudio.com/docs/python/environments).
4. **Run All.**

## Training on your own machine: what changes

Running a model needs its learned numbers in memory. Training needs much more, because for
every learned number the computer also keeps a gradient, which says which way to nudge it.
With the AdamW optimizer it also keeps two running averages.

| At full precision | Bytes per learned number | SmolLM2-135M | Qwen2.5-0.5B | Qwen2.5-1.5B |
| --- | ---: | ---: | ---: | ---: |
| Running it (the numbers only) | 4 | 0.5 GB | 2.0 GB | 6.2 GB |
| Full fine-tuning with AdamW (numbers, gradients, two averages) | 16 | 2.2 GB | 7.9 GB | 24.7 GB |

On top of that comes working memory for the text in each batch, which grows with the batch size
and the length of the texts.

- **That is why Week 6 fine-tunes with LoRA.** The model's own numbers stay frozen and only a
  small add-on is trained, so the gradient and optimizer bills almost disappear
  ([Hugging Face PEFT](https://huggingface.co/docs/peft/index)). QLoRA also stores the frozen
  model in 4 bits; the course runs it on an NVIDIA GPU, either yours or Colab's.
- **On NVIDIA, mixed precision** saves memory and time. Cards with compute capability 8.0 or
  higher (RTX 30-series and newer) handle `bfloat16`: set `bf16=True` in Hugging Face's
  `TrainingArguments`. Older cards and Colab's T4 use `fp16=True`. Check 3 says which your GPU
  is.
- **On a Mac**, Hugging Face's `Trainer` uses the GPU automatically. The whole model has to fit
  in memory: on a Mac, `device_map="auto"` cannot move layers out to the CPU the way it can with
  CUDA. If memory keeps growing when batches vary in length, set `torch_empty_cache_steps`
  ([Hugging Face: training on Apple silicon](https://huggingface.co/docs/transformers/perf_train_special)).
- **Colab's T4** has about 15 GB of GPU memory, more than most laptop GPUs. It is the fallback
  for anything that does not fit.

Check 7 trains SmolLM2-135M for ten steps. It is the smallest possible proof that your machine
can train.

---

# The checks

Run everything below with **Run All**. Each check prints its result as it goes, and nothing
below changes your system. A check that cannot run records a FAIL and lets the rest continue,
so you always reach the summary.

In [ ]:
# --- 0. Setup for the checks: run this first -------------------------------------------------
import os

# Two settings that only count if they exist before torch and Hugging Face are imported:
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")       # Mac: ops with no GPU version run on the CPU
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: silence a harmless cache warning

import gc
import importlib
import importlib.metadata as metadata
import platform
import re
import shutil
import struct
import subprocess
import sys
import time
from pathlib import Path

RESULTS = []          # one row per check: (check, status, detail)


def record(check: str, status: str, detail: str = "") -> None:
    """Keep one result for the summary at the end, and print it now."""
    RESULTS.append((check, status, detail))
    print(f"{status:<4}  {check}: {detail}")


def run(*cmd: str) -> str:
    """Run a system command; return what it printed ('' if it is missing or fails)."""
    try:
        return subprocess.run(list(cmd), capture_output=True, text=True, timeout=60).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return ""


def short(exc: Exception, n: int = 300) -> str:
    """An exception as one line: its type and the start of its message."""
    return f"{type(exc).__name__}: {' '.join(str(exc).split())[:n]}"


IS_MAC = sys.platform == "darwin"
IS_WINDOWS = sys.platform == "win32"
IS_LINUX = sys.platform.startswith("linux")
IN_COLAB = "google.colab" in sys.modules
IN_WSL = IS_LINUX and "microsoft" in platform.release().lower()
MACHINE = platform.machine()
print("ready")

### Check 1 — This machine and this Python

Which Python is running this notebook, and is it the one you meant (fact 1)? On a Mac: Apple
silicon without Rosetta, and macOS 14 or later. Also: memory, and free disk for the models.

In [ ]:
# --- 1. This machine and this Python ---------------------------------------------------------
CHIP, MACOS, ROSETTA = "", "", False
if IS_MAC:
    CHIP = run("sysctl", "-n", "machdep.cpu.brand_string")
    MACOS = run("sw_vers", "-productVersion") or platform.mac_ver()[0]
    ROSETTA = run("sysctl", "-n", "sysctl.proc_translated") == "1"
MACOS_MAJOR = int((MACOS or "0").split(".")[0])

print(f"Operating system : {platform.platform()}" + (f"  ({CHIP}, macOS {MACOS})" if IS_MAC else "")
      + ("  (WSL2)" if IN_WSL else "") + ("  (Google Colab)" if IN_COLAB else ""))
print(f"This Python      : {platform.python_version()} at {sys.executable}")
print()

# 1a. Python version: the course environment is 3.12; current PyTorch needs 3.10 or newer.
if sys.version_info[:2] == (3, 12):
    record("Python version", "PASS", platform.python_version())
elif sys.version_info[:2] >= (3, 10):
    record("Python version", "WARN", f"{platform.python_version()}: works, but the course environment is 3.12")
else:
    record("Python version", "FAIL", f"{platform.python_version()}: current PyTorch needs 3.10 or newer")

# 1b. A 64-bit Python: PyTorch does not install on a 32-bit one.
bits = struct.calcsize("P") * 8
record("64-bit Python", "PASS" if bits == 64 else "FAIL", f"{bits}-bit")

# 1c. A project environment, not a system Python (fact 1).
prefix = Path(sys.prefix)
if IN_COLAB:
    record("Environment", "PASS", "Google Colab")
elif "WindowsApps" in sys.executable:
    record("Environment", "WARN", "the Microsoft Store Python; use the course environment instead (fact 1)")
elif sys.prefix != sys.base_prefix:
    record("Environment", "PASS", f"virtual environment {prefix.name!r} in {prefix.parent}")
elif (prefix / "conda-meta").exists() and prefix.parent.name == "envs":
    record("Environment", "PASS", f"conda environment {prefix.name!r}")
elif (prefix / "conda-meta").exists():
    record("Environment", "WARN", "conda's base environment; create and select `ism6564` (Path A or B)")
else:
    record("Environment", "WARN", "a system Python, not a course environment (fact 1)")

# 1d. The processor. On a Mac: Apple silicon, no Rosetta, macOS 14 or later.
if IS_MAC and ROSETTA:
    record("Processor", "FAIL", "this Python is an Intel build running under Rosetta, so it cannot "
           "use the Mac's GPU (Path A, step 3)")
elif IS_MAC and MACHINE == "arm64":
    record("Processor", "PASS", f"{CHIP}, arm64 Python")
elif IS_MAC:
    record("Processor", "WARN", "an Intel Mac: current PyTorch no longer supports it (Path D)")
elif IS_WINDOWS and MACHINE.upper() == "ARM64":
    record("Processor", "WARN", "Windows on Arm: PyTorch has no GPU support here (Path D)")
else:
    record("Processor", "PASS", MACHINE)
if IS_MAC and (MACHINE == "arm64" or ROSETTA):
    new_enough = MACOS_MAJOR >= 14
    record("macOS version", "PASS" if new_enough else "FAIL", f"macOS {MACOS}" + (
        "" if new_enough else "; the Mac GPU needs macOS 14 or later (Path A, step 1)"))

# 1e. Memory. On a Mac the GPU shares it.
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 2**30
except ImportError:
    RAM_GB = None
if RAM_GB is None:
    record("Memory (RAM)", "INFO", "unknown: psutil is not installed")
elif RAM_GB >= 15:
    record("Memory (RAM)", "PASS", f"{RAM_GB:.0f} GB")
else:
    record("Memory (RAM)", "WARN", f"{RAM_GB:.0f} GB: fine for small models; close other apps before "
           "the 1.5B model (6.2 GB at full precision), or use Colab")

# 1f. Free disk where Hugging Face keeps downloaded models.
try:
    from huggingface_hub import constants
    HF_CACHE = Path(constants.HF_HUB_CACHE)
except Exception:
    HF_CACHE = Path.home() / ".cache" / "huggingface" / "hub"
probe = HF_CACHE
while not probe.exists() and probe != probe.parent:
    probe = probe.parent
free_gb = shutil.disk_usage(probe).free / 2**30
print(f"Model cache      : {HF_CACHE}")
if free_gb >= 20:
    record("Free disk", "PASS", f"{free_gb:.0f} GB free for downloaded models")
elif free_gb >= 8:
    record("Free disk", "WARN", f"{free_gb:.0f} GB free: enough for Assignment 5 (about 6 GB of models), not much more")
else:
    record("Free disk", "FAIL", f"{free_gb:.1f} GB free: Assignment 5 alone downloads about 6 GB")

### Check 2 — The packages

Is each package installed, does it import, and is it new enough? The first six are required.
The rest are for the lab and Week 6; if they are missing, that is a WARN.

In [ ]:
# --- 2. The packages -------------------------------------------------------------------------
def major_minor(version: str) -> tuple:
    """'2.14.0+cu126' -> (2, 14)."""
    m = re.match(r"(\d+)\.(\d+)", version)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)


def check_package(module, pip_name, minimum, why, missing):
    """Installed? Imports? New enough? `missing` is the status to give it if it is absent."""
    try:
        version = metadata.version(pip_name)
    except metadata.PackageNotFoundError:
        record(pip_name, missing, f"not installed ({why})")
        return
    try:
        importlib.import_module(module)
    except Exception as exc:
        hint = ""
        if IS_WINDOWS and ("DLL" in str(exc) or "WinError" in str(exc)):
            hint = " (install the Microsoft Visual C++ Redistributable: Path B)"
        record(pip_name, "FAIL", f"{version} is installed but does not import: {short(exc)}{hint}")
        return
    if minimum and major_minor(version) < minimum:
        record(pip_name, missing, f"{version}; needs {minimum[0]}.{minimum[1]} or newer ({why})")
    else:
        record(pip_name, "PASS", version)


# (import name, pip name, minimum version, what the course uses it for)
REQUIRED = [
    ("torch", "torch", (2, 5), "runs every model; current transformers needs 2.5 or newer"),
    ("transformers", "transformers", (4, 56), "loads models by name; the course code uses dtype=, new in 4.56"),
    ("huggingface_hub", "huggingface-hub", None, "downloads models"),
    ("numpy", "numpy", None, "arrays"),
    ("pandas", "pandas", None, "tables"),
    ("sklearn", "scikit-learn", None, "metrics in Assignment 5"),
]
RECOMMENDED = [
    ("accelerate", "accelerate", None, "training with Hugging Face's Trainer, Week 6"),
    ("peft", "peft", None, "LoRA fine-tuning, Week 6"),
    ("datasets", "datasets", None, "training data, Week 6"),
    ("sentence_transformers", "sentence-transformers", None, "embeddings: Week 5 lab, Week 7"),
    ("matplotlib", "matplotlib", None, "charts"),
]
for row in REQUIRED:
    check_package(*row, missing="FAIL")
for row in RECOMMENDED:
    check_package(*row, missing="WARN")

HAVE_TORCH = "torch" in sys.modules
if HAVE_TORCH:
    import torch
    built_for = (f"NVIDIA GPUs (CUDA {torch.version.cuda})" if torch.version.cuda
                 else "the CPU and the Mac GPU" if IS_MAC else "the CPU only")
    print(f"\nThis PyTorch ({torch.__version__}) is the build for {built_for}.")

### Check 3 — Which GPU can PyTorch use?

The course notebooks pick an NVIDIA GPU (`cuda`) first, then the Mac GPU (`mps`), then the
CPU. If you have a GPU and this check lands on the CPU, it tells you why.

In [ ]:
# --- 3. Which GPU can PyTorch use? -----------------------------------------------------------
device = None
NVIDIA_GPUS = run("nvidia-smi", "-L") if shutil.which("nvidia-smi") else ""    # one line per NVIDIA GPU


def build_covers(major: int, minor: int) -> bool:
    """Does this PyTorch build contain code for a GPU of compute capability major.minor?"""
    for arch in torch.cuda.get_arch_list():
        m = re.fullmatch(r"(sm|compute)_(\d+)(\d)[a-z]?", arch)
        if not m:
            continue
        kind, a_major, a_minor = m.group(1), int(m.group(2)), int(m.group(3))
        if kind == "sm" and a_major == major and a_minor <= minor:
            return True
        if kind == "compute" and (a_major, a_minor) <= (major, minor):
            return True
    return False


if not HAVE_TORCH:
    record("GPU", "FAIL", "PyTorch does not import; fix check 2 first")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    gpu_gb = props.total_memory / 2**30
    driver = run("nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader").splitlines()
    print(f"NVIDIA GPU       : {props.name}, {gpu_gb:.1f} GB, compute capability {props.major}.{props.minor}")
    print(f"Driver, build    : driver {driver[0] if driver else 'unknown'}, PyTorch built for CUDA {torch.version.cuda}")
    record("GPU", "PASS", f"cuda: {props.name}")
    if build_covers(props.major, props.minor):
        record("Build fits the GPU", "PASS", f"compute capability {props.major}.{props.minor} is covered")
    else:
        record("Build fits the GPU", "FAIL", f"this PyTorch has no code for compute capability "
               f"{props.major}.{props.minor}; see the table in Path B, step 3")
    if gpu_gb >= 7.5:
        record("GPU memory", "PASS", f"{gpu_gb:.1f} GB")
    else:
        record("GPU memory", "WARN", f"{gpu_gb:.1f} GB: the 1.5B model at full precision (6.2 GB) will not "
               "fit; use smaller models or Colab")
    record("bfloat16 training", "INFO", "supported (compute capability 8.0+): bf16=True"
           if props.major >= 8 else "not on this GPU: use fp16=True for mixed precision")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    record("GPU", "PASS", f"mps: the {CHIP} GPU, through Metal")
    if hasattr(torch.mps, "recommended_max_memory") and RAM_GB:
        print(f"Mac GPU memory   : up to {torch.mps.recommended_max_memory() / 2**30:.1f} GB of the "
              f"{RAM_GB:.0f} GB that the CPU and GPU share")
else:
    device = torch.device("cpu")
    first_gpu = re.sub(r"^GPU \d+: | \(UUID.*$", "", NVIDIA_GPUS.splitlines()[0]) if NVIDIA_GPUS else ""
    if IN_COLAB:
        status, why = "FAIL", "this Colab session has no GPU: Runtime > Change runtime type > T4 GPU"
    elif IS_MAC and ROSETTA:
        status, why = "FAIL", "an Intel Python under Rosetta cannot use the Mac GPU (Path A, step 3)"
    elif IS_MAC and MACHINE != "arm64":
        status, why = "WARN", "an Intel Mac has no GPU that PyTorch can use (Path D)"
    elif IS_MAC and not torch.backends.mps.is_built():
        status, why = "FAIL", "this PyTorch was built without Mac GPU support; reinstall it (Path A, step 4)"
    elif IS_MAC and MACOS_MAJOR < 14:
        status, why = "FAIL", f"macOS {MACOS} is too old for the Mac GPU; update to 14 or later (Path A, step 1)"
    elif IS_MAC:
        status, why = "FAIL", "PyTorch reports the Mac GPU as unavailable; see Troubleshooting"
    elif first_gpu and torch.version.cuda is None:
        status, why = "FAIL", f"{first_gpu} is present, but this is the CPU-only PyTorch (Path B, step 4)"
    elif first_gpu:
        status, why = "FAIL", (f"PyTorch is built for CUDA {torch.version.cuda} but cannot reach the GPU: "
                               "update the driver, or use the cu126 build (Path B, steps 1-4)")
    elif shutil.which("nvidia-smi"):
        status, why = "FAIL", "nvidia-smi is installed but cannot reach the GPU; reinstall the driver (Path B, step 1)"
    else:
        status, why = "WARN", "no NVIDIA or Apple-silicon GPU found; the CPU works, only more slowly (Path D)"
    record("GPU", status, f"cpu: {why}")

### Check 4 — Can the GPU compute, and does it agree with the CPU?

A GPU that is found but cannot run PyTorch's code fails here. The classic example is an
RTX 50-series card with a `cu126` build.

In [ ]:
# --- 4. Can the GPU compute, and does it agree with the CPU? ---------------------------------
def sync(dev) -> None:
    """Wait until the device has really finished. GPUs work in the background; timing needs this."""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


if device is None:
    record("Compute", "FAIL", "no working PyTorch; fix checks 2 and 3 first")
elif device.type == "cpu":
    record("Compute", "INFO", "running on the CPU, so there is no GPU result to compare")
else:
    try:
        g = torch.Generator().manual_seed(0)
        a, b = torch.randn(512, 512, generator=g), torch.randn(512, 512, generator=g)
        on_cpu = a @ b
        on_gpu = (a.to(device) @ b.to(device)).cpu()
        gap = ((on_gpu - on_cpu).abs().max() / on_cpu.abs().max()).item()
        record("Compute", "PASS" if gap < 1e-3 else "FAIL",
               f"a 512 x 512 matrix product on {device.type} matches the CPU "
               + ("exactly" if gap == 0 else f"to within {gap:.1e} of its largest value"))
    except Exception as exc:
        record("Compute", "FAIL", short(exc))

### Check 5 — How much faster is the GPU?

A large matrix product, the operation language models spend most of their time on, timed on
the CPU and on the GPU.

In [ ]:
# --- 5. How much faster is the GPU? One big matrix product, CPU against GPU ------------------
def gflops(dev, n: int = 2048, reps: int = 5) -> float:
    """Billions of arithmetic operations per second on an n x n matrix product (float32)."""
    a, b = torch.randn(n, n, device=dev), torch.randn(n, n, device=dev)
    a @ b                                   # warm-up: the first call pays one-off start-up costs
    sync(dev)
    t0 = time.perf_counter()
    for _ in range(reps):
        a @ b
    sync(dev)
    return 2 * n**3 * reps / (time.perf_counter() - t0) / 1e9


if device is not None:
    try:
        cpu_speed = gflops(torch.device("cpu"))
        if device.type == "cpu":
            record("Speed", "INFO", f"CPU: {cpu_speed:,.0f} GFLOP/s")
        else:
            gpu_speed = gflops(device)
            ratio = gpu_speed / cpu_speed
            record("Speed", "PASS" if ratio >= 1 else "WARN",
                   f"{device.type}: {gpu_speed:,.0f} GFLOP/s against the CPU's {cpu_speed:,.0f} (x{ratio:.1f})"
                   + ("" if ratio >= 1 else "; slower than the CPU: plug in the charger, check the power mode"))
    except Exception as exc:
        record("Speed", "FAIL", short(exc))

### Check 6 — Download a small language model and have it write

SmolLM2-135M-Instruct (about 270 MB) answers one question, first on the CPU and then on your
GPU, at full precision with greedy decoding (the same prompt always gives the same reply). The
reply should read like English. The two devices should normally pick the same words.

Do not be surprised if the CPU writes *faster* than the GPU here. Each token of a model this
small is too little work to keep a GPU busy, so start-up costs dominate. The GPU pulls ahead as
the models grow: on the instructor's M5 Max it was 1.3 times the CPU's speed on Qwen2.5-0.5B
and 1.7 times on Qwen2.5-1.5B. It also pulls ahead in training (check 7).

In [ ]:
# --- 6. Download a small language model and have it write ------------------------------------
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"        # 135 million learned numbers, about 270 MB
QUESTION = "In one sentence, what does a GPU do?"
GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
model = tok = None

if device is not None:
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        t0 = time.perf_counter()
        tok = AutoTokenizer.from_pretrained(MODEL)
        model = AutoModelForCausalLM.from_pretrained(MODEL).float().eval()   # full precision, on the CPU
        record("Download a model", "PASS", f"{MODEL} ready in {time.perf_counter() - t0:.0f} s "
               "(the first run includes the download)")
    except Exception as exc:
        record("Download a model", "FAIL", short(exc))


def write(dev, n_new: int = 48):
    """Greedy-write exactly n_new tokens on `dev`. Returns (the new token ids, tokens per second)."""
    prompt = tok.apply_chat_template([{"role": "user", "content": QUESTION}], tokenize=False,
                                     add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(dev)
    kw = dict(pad_token_id=tok.eos_token_id, **GREEDY)
    with torch.no_grad():
        model.generate(**ids, max_new_tokens=4, **kw)                  # warm-up
        sync(dev)
        t0 = time.perf_counter()
        out = model.generate(**ids, max_new_tokens=n_new, min_new_tokens=n_new, **kw)
        sync(dev)
        seconds = time.perf_counter() - t0
    return out[0][ids["input_ids"].shape[1]:].cpu(), n_new / seconds


def reads_like_text(text: str) -> bool:
    """A rough test for 'this is language, not garbage'."""
    words = re.findall(r"[A-Za-z]+", text)
    common = {"a", "an", "the", "is", "of", "to", "and", "in", "that", "it", "for", "on", "with"}
    return len(words) >= 5 and any(w.lower() in common for w in words)


if model is not None:
    try:
        cpu_ids, cpu_tps = write(torch.device("cpu"))
        new_ids, tps = cpu_ids, cpu_tps
        if device.type != "cpu":
            model.to(device)
            new_ids, tps = write(device)
        stop = (new_ids == tok.eos_token_id).nonzero()
        reply = tok.decode(new_ids[: stop[0].item()] if len(stop) else new_ids, skip_special_tokens=True).strip()
        print(f"Q: {QUESTION}\nA: {reply}\n")
        sensible = reads_like_text(reply)
        record("Model writes sensibly", "PASS" if sensible else "FAIL",
               f"{len(reply.split())} words of English" if sensible else f"garbled reply: {reply[:80]!r}")
        if device.type == "cpu":
            record("Writing speed", "INFO", f"{cpu_tps:.0f} tokens/s on the CPU")
        else:
            record("Writing speed", "PASS", f"{tps:.0f} tokens/s on {device.type}, {cpu_tps:.0f} on the CPU"
                   + ("" if tps >= cpu_tps else "; the CPU winning is normal for a model this small"))
            same = bool((new_ids[:24] == cpu_ids[:24]).all())
            record("Same words as the CPU", "PASS" if same else "WARN",
                   "the first 24 tokens match" if same else "the reply differs from the CPU's; tiny "
                   "rounding differences can do that, but if the reply reads badly, see Troubleshooting")
    except Exception as exc:
        record("Model writes sensibly", "FAIL", short(exc))

### Check 7 — Train it for ten steps

Full fine-tuning: every one of the model's 135 million learned numbers is nudged, ten times, on
four sentences. The *loss* measures how surprised the model is by the text. If it falls, your
machine can train. On an NVIDIA GPU this check also tries one mixed-precision step.

In [ ]:
# --- 7. Train it for ten steps (full fine-tuning) --------------------------------------------
TEXTS = [
    "Northwind's support desk routes every ticket to one of six queues.",
    "A parcel marked delivered but never received goes to the Shipping queue.",
    "A charge the customer does not recognise goes to the Billing queue.",
    "A product that broke in normal use goes to the Defect queue.",
]

if model is not None:
    try:
        model.to(device).train()
        batch = tok(TEXTS, return_tensors="pt", padding=True).to(device)
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100           # never learn from padding
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        losses = []
        sync(device)
        t0 = time.perf_counter()
        for step in range(10):
            loss = model(**batch, labels=labels).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
        sync(device)
        seconds = time.perf_counter() - t0
        print("loss by step:", " ".join(f"{x:.2f}" for x in losses))
        if device.type == "cuda":
            print(f"GPU memory at the peak: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
        elif device.type == "mps" and hasattr(torch.mps, "driver_allocated_memory"):
            print(f"Mac GPU memory in use: {torch.mps.driver_allocated_memory() / 2**30:.1f} GB")
        learned = all(x == x for x in losses) and losses[-1] < 0.5 * losses[0]    # x == x is False for NaN
        record("Training", "PASS" if learned else "FAIL",
               f"10 steps in {seconds:.1f} s on {device.type}; loss {losses[0]:.2f} -> {losses[-1]:.2f}"
               + ("" if learned else "; the loss should fall by more than half"))
        if device.type == "cuda":
            half = torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16
            with torch.no_grad(), torch.autocast("cuda", dtype=half):
                half_loss = model(**batch, labels=labels).loss.item()
            record("Mixed precision", "PASS" if half_loss == half_loss else "WARN",
                   f"a {str(half).replace('torch.', '')} forward pass gives loss {half_loss:.2f}")
    except Exception as exc:
        record("Training", "FAIL", short(exc))
    finally:
        model = optimizer = None
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        elif device.type == "mps":
            torch.mps.empty_cache()

### Check 8 — Optional, Mac only: Apple's MLX

This check is skipped unless MLX is installed (see Path A). It never FAILs your setup.

In [ ]:
# --- 8. Optional, Mac only: Apple's MLX ------------------------------------------------------
if IS_MAC and MACHINE == "arm64":
    try:
        import mlx.core as mx
        a, b = mx.random.normal((2048, 2048)), mx.random.normal((2048, 2048))
        mx.eval(a @ b)                                     # warm-up; MLX computes only when asked
        t0 = time.perf_counter()
        for _ in range(5):
            mx.eval(a @ b)
        mlx_speed = 2 * 2048**3 * 5 / (time.perf_counter() - t0) / 1e9
        record("MLX (optional)", "PASS", f"mlx {metadata.version('mlx')} on {mx.default_device()}: "
               f"{mlx_speed:,.0f} GFLOP/s")
    except ImportError:
        record("MLX (optional)", "INFO", "not installed; it is optional (Path A)")
    except Exception as exc:
        record("MLX (optional)", "WARN", short(exc))
else:
    print("MLX runs only on Apple silicon; skipped.")

### Summary

In [ ]:
# --- 9. Summary ------------------------------------------------------------------------------
fails = [r for r in RESULTS if r[1] == "FAIL"]
warns = [r for r in RESULTS if r[1] == "WARN"]
width = max(len(r[0]) for r in RESULTS)
for check, status, detail in RESULTS:
    print(f"{status:<4}  {check:<{width}}  {detail}")
print()
if fails:
    print(f"NOT READY: {len(fails)} check(s) failed. Fix them from the top down (a later failure is")
    print("often caused by an earlier one), restart the kernel, and run this notebook again.")
elif warns:
    print("READY, WITH WARNINGS: everything the course needs works. Read each WARN: some only")
    print("mean slower runs or smaller models; others ask you to fix something.")
else:
    print("READY: this machine can run and train language models for this course.")

print("\n----- copy from here when you ask for help (an AI assistant, a classmate) -----")
print(f"OS: {platform.platform()}" + (f" | {CHIP}, macOS {MACOS}" if IS_MAC else ""))
print(f"Python {platform.python_version()} at {sys.executable}")
if HAVE_TORCH:
    print(f"torch {torch.__version__} (CUDA build: {torch.version.cuda}) | device used: {device}")
if NVIDIA_GPUS:
    print("nvidia-smi -L: " + " / ".join(line.split(" (UUID")[0] for line in NVIDIA_GPUS.splitlines()))
for check, status, detail in fails + warns:
    print(f"{status} {check}: {detail}")
print("----- to here -----")

---

## When a check fails

**Troubleshoot like this:**

1. **Read the last line of the error first**, because it names the problem. Then find the
   first line of the traceback that points to your own code or command.
2. **Search the exact message**, in quotes. Prefer the official documentation and the GitHub
   issues of the project that raised it.
3. **Ask an AI assistant.** Paste the summary block above and the full error, and say what you
   have already tried. Check its advice against the official pages before you run anything
   that uninstalls or deletes.
4. **Change one thing at a time.** Restart the kernel, then run this notebook again.
5. **If an evening goes by, stop.** Do the week's work in Colab and come back to the install
   later. Do not let an install problem cost you a deadline.

| What you see | What it usually means | What to do |
| --- | --- | --- |
| `No module named 'torch'` (or another package) here, but it works in your terminal | The notebook's kernel is a different Python (fact 1) | Select your course environment as the kernel, top right. Check 1 prints the Python in use. |
| Mac: check 1 says **Rosetta**, or `platform.machine()` prints `x86_64` on an M-series Mac | An Intel Python | Path A, step 3: remove it, install an Apple-silicon Python, rebuild the environment. |
| Mac: the Mac GPU is unavailable on an M-series Mac | macOS older than 14, or a PyTorch without Mac GPU support | Update macOS (Path A, step 1); reinstall PyTorch (Path A, step 4). |
| Windows: the version has no `+cu…` ending, and check 3 says CPU-only | The CPU-only build | Path B, step 4. |
| Windows: `OSError: [WinError 126]` … `c10.dll` (or another DLL) | A missing Microsoft C++ runtime | Install the Visual C++ Redistributable (x64) and restart (Path B). |
| `no kernel image is available for execution on the device`, or a warning that your GPU's `sm_…` is not compatible | The build has no code for your GPU | RTX 50-series: `cu130`. GTX 10-series and older: `cu126`. (Path B, step 3) |
| `nvidia-smi` is not found, or cannot communicate with the driver | No working NVIDIA driver | Install the newest driver and restart (Path B, step 1). |
| `torch.cuda.is_available()` is `False` with a `+cu130` build | A driver older than 580 | Update the driver, or use `cu126`. |
| `CUDA out of memory` or `MPS backend out of memory` | The model does not fit | Close other programs, restart the kernel, use a smaller model or Colab. On a Mac, do not set `PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0` (Path A). |
| `NotImplementedError: The operator … is not currently implemented for the MPS device` | An operation with no Mac GPU version | Set `PYTORCH_ENABLE_MPS_FALLBACK=1` before `import torch` (check 0 does); restart the kernel. |
| The kernel dies without an error message | Usually out of memory (RAM) | Close other apps, restart, use a smaller model. |
| Downloads fail or stall, or `429 Too Many Requests` | The network, or Hugging Face's rate limit | Run the cell again: downloads resume. Some networks block `huggingface.co`, so try another network or Colab. A free Hugging Face account and token raise the limit. |
| `You are sending unauthenticated requests to the HF Hub` | A notice, not an error | Ignore it, or log in with a free token. |
| `pip` fails with a path error on Windows | The 260-character path limit | Enable long paths, or use a short folder such as `C:\ism6564` (Path B). |
| Windows with uv: the GPU worked before, and now it is the CPU | `./setup.sh`, `uv sync` or `uv run` restored the CPU-only build | Run the uv line in Path B, step 4 again. |
| `conda: command not found` | The terminal started before `conda init` | Close it and open a new one (Windows: *Anaconda Prompt*). |
| VS Code does not list your environment | VS Code has not found it | *Python: Select Interpreter → Enter interpreter path* → the `python` inside your environment, then *Developer: Reload Window*. |

## References

**PyTorch**
- [Start locally: the install selector](https://pytorch.org/get-started/locally/)
- [MPS backend (the Mac GPU)](https://docs.pytorch.org/docs/stable/notes/mps.html) ·
  [MPS environment variables](https://docs.pytorch.org/docs/stable/mps_environment_variables.html)
- [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch 2.7 release: Blackwell (RTX 50-series) and CUDA 12.8](https://pytorch.org/blog/pytorch-2-7/)
- [Introducing accelerated PyTorch training on Mac (2022 background)](https://pytorch.org/blog/introducing-accelerated-pytorch-training-on-mac/)

**Apple**
- [Accelerated PyTorch training on Mac: requirements and install](https://developer.apple.com/metal/pytorch/)
- [MLX](https://github.com/ml-explore/mlx) · [mlx-lm](https://github.com/ml-explore/mlx-lm) ·
  [mlx-lm LoRA and QLoRA guide](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md)
- [Exploring LLMs with MLX and the Neural Accelerators in the M5 GPU](https://machinelearning.apple.com/research/exploring-llms-mlx-m5)

**NVIDIA**
- [Driver downloads](https://www.nvidia.com/en-us/drivers/)
- [CUDA GPUs and their compute capability](https://developer.nvidia.com/cuda-gpus)
- [CUDA Toolkit release notes: minimum driver per CUDA version](https://docs.nvidia.com/cuda/cuda-toolkit-release-notes/index.html)
- [CUDA on WSL user guide](https://docs.nvidia.com/cuda/wsl-user-guide/index.html)
- [CUDA installation guide for Windows (only if you compile GPU code)](https://docs.nvidia.com/cuda/cuda-installation-guide-microsoft-windows/index.html)

**Microsoft**
- [Install WSL](https://learn.microsoft.com/en-us/windows/wsl/install) ·
  [Enable NVIDIA CUDA on WSL](https://learn.microsoft.com/en-us/windows/ai/directml/gpu-cuda-in-wsl)
- [Latest Visual C++ Redistributable](https://learn.microsoft.com/en-us/cpp/windows/latest-supported-vc-redist)
- [Maximum path length limitation (enabling long paths)](https://learn.microsoft.com/en-us/windows/win32/fileio/maximum-file-path-limitation)

**Hugging Face**
- [Transformers installation](https://huggingface.co/docs/transformers/installation)
- [Training on Apple silicon](https://huggingface.co/docs/transformers/perf_train_special)
- [Manage the model cache](https://huggingface.co/docs/huggingface_hub/guides/manage-cache)
- [PEFT (LoRA)](https://huggingface.co/docs/peft/index)
- [SmolLM2-135M-Instruct, the model used by the checks](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)

**Tools**
- [uv: installation](https://docs.astral.sh/uv/getting-started/installation/) ·
  [uv with PyTorch](https://docs.astral.sh/uv/guides/integration/pytorch/)
- [Miniconda](https://www.anaconda.com/docs/getting-started/miniconda/main) ·
  [the course's getting-started guide](https://github.com/prof-tcsmith/6564F26-DATA/blob/main/docs/getting-started-local.md)
- [VS Code: Jupyter notebooks](https://code.visualstudio.com/docs/datascience/jupyter-notebooks) ·
  [Python environments](https://code.visualstudio.com/docs/python/environments) ·
  [WSL](https://code.visualstudio.com/docs/remote/wsl)
- [Google Colab](https://colab.research.google.com/)